In [ ]:
# ============================================================
# FULL MODEL RERUN: CONVNEXT + SWIN + EVA02 + FINAL ENSEMBLE
# Freshly trains all 3 model families again.
#
# Input:
# /content/drive/MyDrive/Kaggle/train
# /content/drive/MyDrive/Kaggle/test
#
# Output:
# /content/drive/MyDrive/Kaggle/full_rerun_mega_ensemble/full_rerun_mega_ensemble_ID_target.csv
# ============================================================

!pip -q install timm scikit-learn

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import json
import math
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy, LabelSmoothingCrossEntropy

from torchvision import transforms
from sklearn.model_selection import StratifiedKFold

from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# PATHS
# ============================================================

TRAIN_DIR = "/content/drive/MyDrive/Kaggle/train"
TEST_DIR  = "/content/drive/MyDrive/Kaggle/test"

SAVE_DIR = "/content/drive/MyDrive/Kaggle/full_rerun_mega_ensemble"
LOCAL_DIR = "/content/full_rerun_mega_ensemble"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

FINAL_CSV_PATH = os.path.join(SAVE_DIR, "full_rerun_mega_ensemble_ID_target.csv")
FINAL_PROBS_PATH = os.path.join(SAVE_DIR, "full_rerun_mega_ensemble_probs.npy")
INFO_PATH = os.path.join(SAVE_DIR, "full_rerun_mega_ensemble_info.json")

# ============================================================
# CONFIG
# ============================================================

SEED = 42
NUM_CLASSES = 100
N_FOLDS = 4

# Set this lower if you want a quicker test run.
FINAL_EPOCHS = 120

# H100/A100 safe defaults.
# If OOM, lower EVA batch first.
BATCH_CONVNEXT = 64
BATCH_SWIN = 48
BATCH_EVA = 32

ACCUM_CONVNEXT = 1
ACCUM_SWIN = 1
ACCUM_EVA = 3

PRED_BATCH = 64
NUM_WORKERS = 4

USE_MIXUP = True
USE_AMP = True
USE_TTA_FLIP = True

# Final ensemble weights.
# EVA highest because your EVA02 accuracy run was strongest.
ENSEMBLE_WEIGHTS = {
    "convnext": 1.15,
    "swin": 1.00,
    "eva02": 1.35,
}

RUN_CONVNEXT = True
RUN_SWIN = True
RUN_EVA02 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Train dir exists:", os.path.exists(TRAIN_DIR))
print("Test dir exists:", os.path.exists(TEST_DIR))

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)

# ============================================================
# SEED
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# ============================================================
# DATA
# ============================================================

def get_train_items(train_dir):
    items = []
    for label in sorted(os.listdir(train_dir), key=lambda x: int(x) if x.isdigit() else x):
        class_dir = os.path.join(train_dir, label)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                items.append((os.path.join(class_dir, fname), int(label)))
    return items

def get_test_items(test_dir):
    files = []
    for fname in os.listdir(test_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            files.append(fname)
    files = sorted(
        files,
        key=lambda x: int(os.path.splitext(x)[0]) if os.path.splitext(x)[0].isdigit() else x
    )
    return [(os.path.join(test_dir, f), f) for f in files]

train_items = get_train_items(TRAIN_DIR)
test_items = get_test_items(TEST_DIR)
labels = np.array([y for _, y in train_items])

print("Train images:", len(train_items))
print("Test images:", len(test_items))
print("Classes:", len(set(labels)))
print("Smallest class:", pd.Series(labels).value_counts().min())
print("Largest class:", pd.Series(labels).value_counts().max())

assert len(train_items) > 0, "No train images found."
assert len(test_items) > 0, "No test images found."
assert len(set(labels)) == NUM_CLASSES, "Expected 100 classes."

# ============================================================
# DATASET
# ============================================================

class ImageDataset(Dataset):
    def __init__(self, items, transform=None, test=False):
        self.items = items
        self.transform = transform
        self.test = test

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label_or_id = self.items[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label_or_id

# ============================================================
# TRANSFORMS
# ============================================================

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def make_transforms(img_size, crop_min, rot_deg, color_strength, rand_erasing):
    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(crop_min, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(rot_deg),
        transforms.ColorJitter(
            brightness=color_strength,
            contrast=color_strength,
            saturation=color_strength,
            hue=0.03
        ),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=rand_erasing, scale=(0.02, 0.18), ratio=(0.3, 3.3)),
    ])

    val_tfms = transforms.Compose([
        transforms.Resize(int(img_size * 1.10)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    test_tfms = val_tfms

    return train_tfms, val_tfms, test_tfms

# ============================================================
# MODEL NAME RESOLVER
# ============================================================

def resolve_model_name(patterns):
    all_models = set(timm.list_models(pretrained=True))
    for p in patterns:
        matches = sorted([m for m in all_models if p in m])
        if matches:
            print("Resolved", p, "->", matches[0])
            return matches[0]
    raise ValueError(f"No timm model found for patterns: {patterns}")

MODEL_NAMES = {
    "convnext": resolve_model_name([
        "convnext_large.fb_in22k_ft_in1k",
        "convnext_large",
        "convnext_base.fb_in22k_ft_in1k",
        "convnext_base",
    ]),
    "swin": resolve_model_name([
        "swin_base_patch4_window12_384.ms_in22k_ft_in1k",
        "swin_base_patch4_window12_384",
        "swin_base",
    ]),
    "eva02": resolve_model_name([
        "eva02_large_patch14_448.mim_m38m_ft_in22k_ft_in1k",
        "eva02_large_patch14_448",
        "eva02_large",
    ]),
}

print("Model names:", MODEL_NAMES)

# ============================================================
# PARAMS
# ============================================================

PARAMS = {
    "convnext": {
        "img_size": 384,
        "batch_size": BATCH_CONVNEXT,
        "accum_steps": ACCUM_CONVNEXT,
        "lr": 1.15e-4,
        "weight_decay": 0.045,
        "drop_rate": 0.10,
        "drop_path_rate": 0.12,
        "mixup_alpha": 0.12,
        "cutmix_alpha": 0.8,
        "mixup_prob": 0.50,
        "label_smoothing": 0.06,
        "rand_erasing": 0.21,
        "rot_deg": 6,
        "color_strength": 0.26,
        "crop_min": 0.58,
        "warmup_epochs": 4,
    },
    "swin": {
        "img_size": 384,
        "batch_size": BATCH_SWIN,
        "accum_steps": ACCUM_SWIN,
        "lr": 8.5e-5,
        "weight_decay": 0.055,
        "drop_rate": 0.10,
        "drop_path_rate": 0.15,
        "mixup_alpha": 0.12,
        "cutmix_alpha": 0.6,
        "mixup_prob": 0.50,
        "label_smoothing": 0.07,
        "rand_erasing": 0.24,
        "rot_deg": 6,
        "color_strength": 0.20,
        "crop_min": 0.60,
        "warmup_epochs": 4,
    },
    "eva02": {
        "img_size": 448,
        "batch_size": BATCH_EVA,
        "accum_steps": ACCUM_EVA,
        "lr": 9.13e-5,
        "weight_decay": 0.075,
        "drop_rate": 0.166,
        "drop_path_rate": 0.175,
        "mixup_alpha": 0.03,
        "cutmix_alpha": 0.5,
        "mixup_prob": 0.545,
        "label_smoothing": 0.04,
        "rand_erasing": 0.30,
        "rot_deg": 6,
        "color_strength": 0.17,
        "crop_min": 0.62,
        "warmup_epochs": 4,
    },
}

# ============================================================
# MODEL / TRAINING HELPERS
# ============================================================

def make_model(family):
    p = PARAMS[family]
    model = timm.create_model(
        MODEL_NAMES[family],
        pretrained=True,
        num_classes=NUM_CLASSES,
        drop_rate=p["drop_rate"],
        drop_path_rate=p["drop_path_rate"],
    )
    return model

def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()

def make_scheduler(optimizer, total_steps, warmup_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion, mixup_fn, accum_steps):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_n = 0

    optimizer.zero_grad(set_to_none=True)

    for step, (x, y) in enumerate(loader):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        y_for_acc = y.clone()

        if mixup_fn is not None:
            x, y_mixed = mixup_fn(x, y)
        else:
            y_mixed = y

        with autocast(device_type="cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
            loss = criterion(logits, y_mixed)
            loss = loss / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        batch_size = x.size(0)
        total_loss += loss.item() * accum_steps * batch_size
        total_correct += (logits.detach().argmax(dim=1) == y_for_acc).sum().item()
        total_n += batch_size

    return total_loss / total_n, total_correct / total_n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_n = 0

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with autocast(device_type="cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_n += batch_size

    return total_loss / total_n, total_correct / total_n

def cleanup():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# ============================================================
# TRAIN FAMILY
# ============================================================

def train_family(family):
    print("\n" + "=" * 90)
    print("TRAINING FAMILY:", family.upper())
    print("=" * 90)

    p = PARAMS[family]
    family_dir = os.path.join(SAVE_DIR, family)
    os.makedirs(family_dir, exist_ok=True)

    train_tfms, val_tfms, test_tfms = make_transforms(
        img_size=p["img_size"],
        crop_min=p["crop_min"],
        rot_deg=p["rot_deg"],
        color_strength=p["color_strength"],
        rand_erasing=p["rand_erasing"],
    )

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    fold_ckpts = []
    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), start=1):
        print("\n" + "-" * 90)
        print(f"{family.upper()} FOLD {fold}/{N_FOLDS}")
        print("-" * 90)

        seed_everything(SEED + fold)

        tr_items = [train_items[i] for i in train_idx]
        va_items = [train_items[i] for i in val_idx]

        tr_ds = ImageDataset(tr_items, transform=train_tfms, test=False)
        va_ds = ImageDataset(va_items, transform=val_tfms, test=False)

        tr_loader = DataLoader(
            tr_ds,
            batch_size=p["batch_size"],
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            drop_last=True,
            persistent_workers=(NUM_WORKERS > 0),
        )

        va_loader = DataLoader(
            va_ds,
            batch_size=PRED_BATCH,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=(NUM_WORKERS > 0),
        )

        model = make_model(family).to(DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=p["lr"],
            weight_decay=p["weight_decay"],
        )

        total_update_steps = math.ceil(len(tr_loader) / p["accum_steps"]) * FINAL_EPOCHS
        warmup_steps = math.ceil(len(tr_loader) / p["accum_steps"]) * p["warmup_epochs"]
        scheduler = make_scheduler(optimizer, total_update_steps, warmup_steps)

        scaler = GradScaler(enabled=(USE_AMP and DEVICE == "cuda"))

        if USE_MIXUP:
            mixup_fn = Mixup(
                mixup_alpha=p["mixup_alpha"],
                cutmix_alpha=p["cutmix_alpha"],
                prob=p["mixup_prob"],
                switch_prob=0.5,
                mode="batch",
                label_smoothing=p["label_smoothing"],
                num_classes=NUM_CLASSES,
            )
            train_criterion = SoftTargetCrossEntropy()
        else:
            mixup_fn = None
            train_criterion = LabelSmoothingCrossEntropy(smoothing=p["label_smoothing"])

        val_criterion = nn.CrossEntropyLoss()

        best_acc = -1.0
        best_loss = 999.0
        best_epoch = -1
        bad_epochs = 0
        patience = 12

        ckpt_path = os.path.join(family_dir, f"{family}_fold{fold}_best.pt")

        for epoch in range(1, FINAL_EPOCHS + 1):
            t0 = time.time()

            train_loss, train_acc = train_one_epoch(
                model=model,
                loader=tr_loader,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                criterion=train_criterion,
                mixup_fn=mixup_fn,
                accum_steps=p["accum_steps"],
            )

            val_loss, val_acc = evaluate(model, va_loader, val_criterion)
            gap = train_acc - val_acc

            improved = val_acc > best_acc

            if improved:
                best_acc = val_acc
                best_loss = val_loss
                best_epoch = epoch
                bad_epochs = 0

                torch.save(
                    {
                        "family": family,
                        "model_name": MODEL_NAMES[family],
                        "params": p,
                        "fold": fold,
                        "epoch": epoch,
                        "best_acc": best_acc,
                        "best_loss": best_loss,
                        "model_state_dict": model.state_dict(),
                    },
                    ckpt_path,
                )
                print("Saved new best:", ckpt_path)
            else:
                bad_epochs += 1

            dt = time.time() - t0

            print(
                f"{family} | Fold {fold} | Epoch {epoch}/{FINAL_EPOCHS} | "
                f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
                f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
                f"Gap: {gap:.4f} | Best: {best_acc:.4f} @ {best_epoch} | "
                f"Bad: {bad_epochs}/{patience} | {dt:.1f}s"
            )

            if DEVICE == "cuda":
                print(
                    "VRAM allocated:",
                    round(torch.cuda.memory_allocated() / 1024**3, 2),
                    "GB | reserved:",
                    round(torch.cuda.memory_reserved() / 1024**3, 2),
                    "GB | peak:",
                    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
                    "GB"
                )

            if bad_epochs >= patience:
                print("Early stopping.")
                break

        fold_ckpts.append(ckpt_path)
        fold_metrics.append({
            "fold": fold,
            "best_acc": float(best_acc),
            "best_loss": float(best_loss),
            "best_epoch": int(best_epoch),
            "ckpt": ckpt_path,
        })

        del model, optimizer, scheduler, scaler, tr_loader, va_loader, tr_ds, va_ds
        cleanup()

    with open(os.path.join(family_dir, f"{family}_metrics.json"), "w") as f:
        json.dump(fold_metrics, f, indent=2)

    print("\nFinished", family)
    print(pd.DataFrame(fold_metrics))

    return fold_ckpts, test_tfms, fold_metrics

# ============================================================
# PREDICT FAMILY
# ============================================================

@torch.no_grad()
def predict_checkpoint(family, ckpt_path, test_tfms):
    p = PARAMS[family]

    test_ds = ImageDataset(test_items, transform=test_tfms, test=True)
    test_loader = DataLoader(
        test_ds,
        batch_size=PRED_BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )

    model = make_model(family).to(DEVICE)

    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    all_probs = []
    all_ids = []

    for x, ids in test_loader:
        x = x.to(DEVICE, non_blocking=True)

        with autocast(device_type="cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)

            if USE_TTA_FLIP:
                logits_flip = model(torch.flip(x, dims=[3]))
                logits = (logits + logits_flip) / 2.0

        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        all_probs.append(probs)
        all_ids.extend(list(ids))

    probs = np.concatenate(all_probs, axis=0)

    del model, test_loader, test_ds
    cleanup()

    return probs, all_ids

def predict_family(family, ckpts, test_tfms):
    print("\n" + "=" * 90)
    print("PREDICTING FAMILY:", family.upper())
    print("=" * 90)

    family_probs = []
    final_ids = None

    for ckpt_path in ckpts:
        print("Predicting:", ckpt_path)
        probs, ids = predict_checkpoint(family, ckpt_path, test_tfms)

        if final_ids is None:
            final_ids = ids
        else:
            assert final_ids == ids, "Test ID order mismatch."

        family_probs.append(probs)

    family_probs = np.mean(family_probs, axis=0)
    family_probs = np.clip(family_probs, 0, None)
    family_probs = family_probs / family_probs.sum(axis=1, keepdims=True)

    prob_path = os.path.join(SAVE_DIR, family, f"{family}_fresh_probs.npy")
    csv_path = os.path.join(SAVE_DIR, family, f"{family}_fresh_ID_target.csv")

    np.save(prob_path, family_probs)

    family_preds = family_probs.argmax(axis=1).astype(int)
    pd.DataFrame({"ID": final_ids, "target": family_preds}).to_csv(csv_path, index=False)

    print("Saved family probs:", prob_path)
    print("Saved family CSV:", csv_path)

    return family_probs, final_ids, prob_path, csv_path

# ============================================================
# RUN FULL TRAINING + PREDICTION
# ============================================================

all_family_outputs = {}
all_metrics = {}

if RUN_CONVNEXT:
    ckpts, test_tfms, metrics = train_family("convnext")
    probs, ids, prob_path, csv_path = predict_family("convnext", ckpts, test_tfms)
    all_family_outputs["convnext"] = {
        "probs": probs,
        "ids": ids,
        "prob_path": prob_path,
        "csv_path": csv_path,
        "ckpts": ckpts,
    }
    all_metrics["convnext"] = metrics

if RUN_SWIN:
    ckpts, test_tfms, metrics = train_family("swin")
    probs, ids, prob_path, csv_path = predict_family("swin", ckpts, test_tfms)
    all_family_outputs["swin"] = {
        "probs": probs,
        "ids": ids,
        "prob_path": prob_path,
        "csv_path": csv_path,
        "ckpts": ckpts,
    }
    all_metrics["swin"] = metrics

if RUN_EVA02:
    ckpts, test_tfms, metrics = train_family("eva02")
    probs, ids, prob_path, csv_path = predict_family("eva02", ckpts, test_tfms)
    all_family_outputs["eva02"] = {
        "probs": probs,
        "ids": ids,
        "prob_path": prob_path,
        "csv_path": csv_path,
        "ckpts": ckpts,
    }
    all_metrics["eva02"] = metrics

# ============================================================
# FINAL ENSEMBLE
# ============================================================

print("\n" + "=" * 90)
print("FINAL ENSEMBLE")
print("=" * 90)

assert len(all_family_outputs) > 0, "No models were run."

final_ids = None
ensemble_probs = None
total_weight = 0.0

for family, out in all_family_outputs.items():
    ids = out["ids"]
    probs = out["probs"]
    w = ENSEMBLE_WEIGHTS[family]

    if final_ids is None:
        final_ids = ids
        ensemble_probs = np.zeros_like(probs, dtype=np.float64)
    else:
        assert final_ids == ids, f"ID mismatch for {family}"

    ensemble_probs += probs * w
    total_weight += w

    print(f"Added {family} with weight {w}")

ensemble_probs /= total_weight
ensemble_probs = np.clip(ensemble_probs, 0, None)
ensemble_probs = ensemble_probs / ensemble_probs.sum(axis=1, keepdims=True)

final_preds = ensemble_probs.argmax(axis=1).astype(int)

submission = pd.DataFrame({
    "ID": final_ids,
    "target": final_preds,
})

submission.to_csv(FINAL_CSV_PATH, index=False)
np.save(FINAL_PROBS_PATH, ensemble_probs)

info = {
    "final_csv": FINAL_CSV_PATH,
    "final_probs": FINAL_PROBS_PATH,
    "num_train": len(train_items),
    "num_test": len(test_items),
    "num_classes": NUM_CLASSES,
    "n_folds": N_FOLDS,
    "final_epochs": FINAL_EPOCHS,
    "use_tta_flip": USE_TTA_FLIP,
    "ensemble_weights": ENSEMBLE_WEIGHTS,
    "model_names": MODEL_NAMES,
    "params": PARAMS,
    "metrics": all_metrics,
    "outputs": {
        family: {
            "prob_path": out["prob_path"],
            "csv_path": out["csv_path"],
            "ckpts": out["ckpts"],
        }
        for family, out in all_family_outputs.items()
    }
}

with open(INFO_PATH, "w") as f:
    json.dump(info, f, indent=2)

print("\nFinal CSV preview:")
print(submission.head())

print("\nClass count preview:")
print(submission["target"].value_counts().sort_index().head(30))

print("\nSaved final CSV:", FINAL_CSV_PATH)
print("Saved final probs:", FINAL_PROBS_PATH)
print("Saved info:", INFO_PATH)
print("DONE.")